# 🐕 Quadruped RL Locomotion — Kaggle Training Notebook

Train a Unitree Go2 quadruped robot to walk using **PPO** in **MuJoCo**.

**Requirements**: Kaggle GPU (T4) or Google Colab

---

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 stable-baselines3>=2.2.0
!pip install -q imageio imageio-ffmpeg pyyaml scipy matplotlib tensorboard

# Clone the project
!git clone https://github.com/langchengg/quadruped-rl-locomotion
%cd quadruped-rl-locomotion

import sys
sys.path.insert(0, '.')

print('✅ Setup complete!')

## 2. Environment Verification

Let's verify the environment loads correctly and test it with a random policy.

In [ ]:
import numpy as np
import mujoco
from src.envs.quadruped_env import QuadrupedEnv, make_quadruped_env

# Create environment
env = make_quadruped_env(render_mode='rgb_array')
obs, info = env.reset()

print(f'✅ Environment created successfully!')
print(f'   Observation shape: {obs.shape}')
print(f'   Action space: {env.action_space}')
print(f'   Commands: {info["commands"]}')

# Run a few random steps
total_reward = 0
for i in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        obs, info = env.reset()

print(f'   Random policy avg reward: {total_reward/100:.4f}')
env.close()

### Visualize Random Policy

In [ ]:
import imageio
from IPython.display import Image, display

env = make_quadruped_env(render_mode='rgb_array')
obs, _ = env.reset()

frames = []
for i in range(200):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    frame = env.render()
    if frame is not None:
        frames.append(frame)
    if terminated or truncated:
        obs, _ = env.reset()

# Save and display GIF
imageio.mimsave('random_policy.gif', frames[::2], duration=50, loop=0)
display(Image(filename='random_policy.gif'))
env.close()
print(f'Captured {len(frames)} frames')

## 3. Train with PPO

Train the Go2 robot using Proximal Policy Optimization.

**Estimated time**: ~2-3 hours on Kaggle T4 GPU for 2M steps

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

import os
os.makedirs('logs/kaggle_ppo', exist_ok=True)

# Create vectorized training environment
def make_env(seed):
    def _init():
        e = make_quadruped_env(render_mode=None)
        e = Monitor(e)
        e.reset(seed=seed)
        return e
    return _init

NUM_ENVS = 4  # Increase to 8-16 on GPU if memory allows
train_env = DummyVecEnv([make_env(i) for i in range(NUM_ENVS)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True)

# Evaluation environment
eval_env = DummyVecEnv([make_env(99)])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False)

print(f'✅ Created {NUM_ENVS} parallel environments')

In [ ]:
# Configure PPO
TOTAL_TIMESTEPS = 2_000_000  # Reduce to 500K for quick test

model = PPO(
    'MlpPolicy',
    train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    max_grad_norm=1.0,
    policy_kwargs={
        'net_arch': dict(pi=[256, 256, 128], vf=[256, 256, 128]),
    },
    verbose=1,
    tensorboard_log='logs/kaggle_ppo',
    device='auto',
    seed=42,
)

# Callbacks
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path='logs/kaggle_ppo',
    eval_freq=max(10000 // NUM_ENVS, 1),
    n_eval_episodes=5,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(100000 // NUM_ENVS, 1),
    save_path='logs/kaggle_ppo',
)

print(f'✅ PPO model created, training for {TOTAL_TIMESTEPS:,} timesteps...')

In [ ]:
# 🚀 Train!
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_callback, checkpoint_callback],
    progress_bar=True,
)

# Save final model
model.save('logs/kaggle_ppo/final_model')
train_env.save('logs/kaggle_ppo/vec_normalize.pkl')
print('✅ Training complete! Model saved.')

## 4. Evaluate Trained Policy

In [ ]:
# Load best model
best_model = PPO.load('logs/kaggle_ppo/best_model')

# Create evaluation environment
eval_env_render = make_quadruped_env(render_mode='rgb_array')
obs, info = eval_env_render.reset()

# Set walking command
eval_env_render.commands[0] = 1.0  # Forward velocity
eval_env_render.commands[1] = 0.0  # Lateral velocity
eval_env_render.commands[2] = 0.0  # Yaw rate

frames = []
total_reward = 0

for step in range(500):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env_render.step(action)
    total_reward += reward
    
    frame = eval_env_render.render()
    if frame is not None:
        frames.append(frame)
    
    if terminated or truncated:
        break

print(f'Episode length: {step+1} steps')
print(f'Total reward: {total_reward:.2f}')
print(f'Average reward: {total_reward/(step+1):.4f}')

# Save walking GIF
os.makedirs('results/videos', exist_ok=True)
imageio.mimsave('results/videos/go2_walking.gif', frames[::2], duration=50, loop=0)
display(Image(filename='results/videos/go2_walking.gif'))

eval_env_render.close()

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load evaluations
try:
    results = np.load('logs/kaggle_ppo/evaluations.npz')
    timesteps = results['timesteps']
    rewards = results['results'].mean(axis=1)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(timesteps, rewards, linewidth=2, color='#2196F3', label='Mean Reward')
    ax.fill_between(timesteps, rewards - results['results'].std(axis=1),
                    rewards + results['results'].std(axis=1), alpha=0.15, color='#2196F3')
    ax.set_xlabel('Timesteps', fontsize=12)
    ax.set_ylabel('Mean Episode Reward', fontsize=12)
    ax.set_title('PPO Training Progress — Go2 Locomotion', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('results/training_curves/ppo_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Training curve saved to results/training_curves/ppo_training.png')
except FileNotFoundError:
    print('⚠️  No evaluation results found. Train the model first.')

## 6. Different Velocity Commands

Test the trained policy with different speed commands.

In [ ]:
commands_to_test = [
    {'name': 'slow_walk',  'vx': 0.3, 'vy': 0.0, 'wz': 0.0},
    {'name': 'fast_walk',  'vx': 1.5, 'vy': 0.0, 'wz': 0.0},
    {'name': 'lateral',    'vx': 0.0, 'vy': 0.3, 'wz': 0.0},
    {'name': 'turning',    'vx': 0.5, 'vy': 0.0, 'wz': 0.5},
]

for cmd in commands_to_test:
    env = make_quadruped_env(render_mode='rgb_array')
    obs, _ = env.reset()
    env.commands[0] = cmd['vx']
    env.commands[1] = cmd['vy']
    env.commands[2] = cmd['wz']
    
    frames = []
    for step in range(300):
        action, _ = best_model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(action)
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        if terminated or truncated:
            break
    
    gif_path = f'results/videos/{cmd["name"]}.gif'
    imageio.mimsave(gif_path, frames[::2], duration=50, loop=0)
    print(f'✅ {cmd["name"]}: {step+1} steps, saved to {gif_path}')
    env.close()

---

## 📦 Download Results

Download your trained model and results for GitHub.

In [ ]:
import shutil

# Package results for download
shutil.make_archive('quadruped_results', 'zip', '.', 'results')
shutil.make_archive('quadruped_model', 'zip', '.', 'logs/kaggle_ppo')

print('✅ Created:')
print('   - quadruped_results.zip (videos, plots)')
print('   - quadruped_model.zip (trained model)')
print('\nDownload these from Files panel →')